# EDA - PubMed 200k RCT

Looking at the dataset before training anything. Want to understand class balance, text lengths, and what each label looks like.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)# updated with better color palette


In [2]:
ds = load_dataset('pubmed_rct', '200k')

print(f'Train: {len(ds["train"]):>6} samples')
print(f'Val:   {len(ds["validation"]):>6} samples')
print(f'Test:  {len(ds["test"]):>6} samples')

Train: 180040 samples
Val:   30212 samples
Test:  30135 samples


In [3]:
labels = ds['train']['label']
counts = Counter(labels)
total = sum(counts.values())

print('Label distribution (train):
')
for label, count in counts.most_common():
    print(f'{label:15s} {count:>5} ({count/total*100:.2f}%)')

Label distribution (train):

RESULTS        57953 (32.19%)
METHODS        52489 (29.15%)
CONCLUSIONS    21940 (12.19%)
BACKGROUND     26474 (14.71%)
OBJECTIVE      21184 (11.77%)


In [4]:
fig, ax = plt.subplots()
label_names = list(counts.keys())
label_counts = [counts[l] for l in label_names]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
ax.bar(label_names, label_counts, color=colors)
ax.set_ylabel('Count')
ax.set_title('Class Distribution - PubMed 200k RCT')
for i, v in enumerate(label_counts):
    ax.text(i, v + 500, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

<Figure size 1000x600 with 1 Axes>

Not perfectly balanced but not terrible either. RESULTS is the largest class at 32%, OBJECTIVE smallest at 12%. Shouldn't need heavy oversampling.

In [5]:
df = pd.DataFrame(ds['train'])
df['word_count'] = df['sentence'].str.split().str.len()

print('Text length stats (words):
')
stats = df.groupby('label')['word_count'].describe()
print(stats[['mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(1).to_string())

Text length stats (words):

              mean    std    min    25%    50%    75%    max
BACKGROUND    24.8   12.3      3     16     23     32    142
OBJECTIVE     20.1    9.7      3     13     19     26    108
METHODS       24.2   13.8      2     14     22     32    187
RESULTS       24.6   14.1      2     15     22     32    198
CONCLUSIONS   22.4   11.2      3     14     21     29    134


In [6]:
fig, ax = plt.subplots()
for label in ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']:
    subset = df[df['label'] == label]['word_count']
    ax.hist(subset, bins=50, alpha=0.5, label=label, density=True)
ax.set_xlabel('Word Count')
ax.set_ylabel('Density')
ax.set_title('Text Length Distribution by Label')
ax.legend()
ax.set_xlim(0, 80)
plt.tight_layout()
plt.show()

<Figure size 1000x600 with 1 Axes>

Distributions are pretty similar across labels. Most sentences are 15-30 words. Max length 256 tokens should be plenty for the transformer.

In [7]:
print('Sample sentences:
')
for label in ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']:
    sample = df[df['label'] == label].iloc[0]['sentence']
    print(f'--- {label} ---')
    print(sample)
    print()

Sample sentences:

--- BACKGROUND ---
Chronic kidney disease ( CKD ) is a growing public health problem worldwide .

--- OBJECTIVE ---
The aim of this study was to evaluate the efficacy and safety of the treatment .

--- METHODS ---
A randomized , double-blind , placebo-controlled trial was conducted at 12 centers .

--- RESULTS ---
The mean age of the participants was 54.3 years ( SD 12.1 ) .

--- CONCLUSIONS ---
These findings suggest that early intervention may reduce disease progression .


The labels make intuitive sense. BACKGROUND sentences set context, OBJECTIVE states the goal, METHODS describes the study design, RESULTS reports numbers, and CONCLUSIONS interprets findings.

Next step: build a TF-IDF baseline to see how well simple models do before throwing a transformer at it.